# Lab: Demand Estimation and Industrial Organization

## Bluetooth Speakers
- The electronics product that had the nicest characteristics for our lab is: Bluetooth Speakers
- The top 5 brands are Bose, JBL, Anker, DOSS, OontZ, plus the **outside option** of Other, which are a collection of small and cheap brands with typically bad reviews
- We want to look at sales data (prices, choices) and infer whether the market is performing well for consumers: Is it competitive? Are prices reasonable or not, relative to cost?

## 1. EDA

Load `demand_2.parquet`. This is brand-level data, which is already consolidated from the many TB of Amazon consumer level data.

- Make kernel density plots for the price and average rating of the products, by brand, as well as describe tables grouped by brand. Which are most expensive? Which are most popular?
- Make plots of the sales, average prices, and market shares of each brand in each year

## Demand
- We typically think that product characteristics like price and features determines which product consumers purchase
- We don't observe the benefit or utility that corresponds to how much consumers like products, just the product they ultimately purchase; this is the latent variable being modeled
- This lends itself really well to multinomial logistic regression (but with a twist)
- This kind of model appears in economics in **industrial organization** and in business as **quantitative marketing**

## Demand Systems
- Imagine consumer $i$ deciding between different brands $j=1, 2, ... , J$ in period $t$, with prices $p_{j}$ and characteristics $x_{jt}$
- The **mean utility** provided by product $j$ to consumer $i$ is
$$
\delta_{jt} = b_0  + \sum_{\ell =1}^L b_\ell x_{jt\ell} + b_{\text{price}} p_{jt}
$$
- So consumers get an average benefit that depends on product characteristics, and then the price reduces that benefit
- When we add an error term and assume that it has the logistic distribution, this becomes multinomial logistic regression: Consumer $i$ gets a shock $\varepsilon_{ijt}$, and gets utilities for each brand $j$ of
$$
U_{ijt} = \delta_{ijt} + \varepsilon_{ijt} = b_0  + \sum_{\ell =1}^L b_\ell x_{jt\ell} + b_{\text{price}} p_{jt}+ \varepsilon_{ijt}
$$

## Demand Systems
- The product with the highest $U_{ijt}$ is the one they purchase, but market shares are determined by
$$
s_{jt} = \begin{cases} 
\frac{\exp(\delta_{jt})}{1 + \sum_{k=1}^{K-1} \exp(\delta_{kt})}, & k = 1, ..., K \\
\frac{1}{1 + \sum_{k=1}^{K-1}\exp(\delta_{kt})}, & \text{outside option}
\end{cases}
$$
- This is great: We just collect product and consumption data, and we can infer demand
- You *could* run this on the original consumer dataset with `.LogisticRegression()`, but that would be very unstable and take forever to converge

## Log-Linearizing the Demand System
- It turns out that estimating demand systems from millions of customers is extremely unstable
- Instead, we'll use a trick to reduce multinomial logistic regression to linear regression
- For the outside option, set $\delta_{Kt} = 0$; this is allowed, because we don't know the scale of the unobserved latent variable anyway -- we're just normalizing the "units" of the unobserved latent scale
- Now, notice that if we divide the share of any brand by demand for the outside option, we get
$$
\dfrac{s_{jt}}{s_{Kt}} = \frac{\exp(\delta_{jt})}{\exp(\delta_{Kt})}
$$
- Now take logs:
$$
\log(s_{jt}) - \log(s_{Kt}) = \delta_{jt} - \underbrace{\delta_{Kt}}_{=0}
$$


## Log-Linearizing the Demand System
- So, we can convert from consumer-level data to brand-level data, and estimate demand using product shares instead of individual consumer choices:
$$
\underbrace{\log(s_{jt}) - \log(s_{Kt})}_{\text{Difference in Log Shares}} = \underbrace{b_0  + \sum_{\ell =1}^L b_\ell x_{jt\ell} + b_{\text{price}} p_{jt}}_{\text{Linear Model}}
$$
- The regression we want to run is:
$$
\underbrace{\log(s_{jt}) - \log(s_{Kt})}_{\text{Difference in Log Shares}} = \underbrace{b_0  + \sum_{\ell =1}^L b_\ell x_{jt\ell} +  b_{\text{price}} p_{jt}}_{\text{Linear Model}}
$$
- So we just compute the log share differences on the left, and use it as our usual $y_{jt}$, and feature engineer the right-hand side
- Learning about consumer preferences and product positioning is **quantitative marketing**

## 2. Estimate the Demand System

Load `demand_3.parquet`. This is brand-level data, which is already consolidated from the consumer level data and aggregated across products sold by that brand.

- The log share differences are already computed, as `log_share_diff`
- Regress those values on price, waterproof flag, party flag, voice assistant flag, year dummies, average rating, and brand dummies

1. What is the coefficient on price? Is it negative? If the price is/were positive, does that make sense? Why might that happen?
2. Which brands have the highest coefficients?
3. Which features (waterproof, voice assistant, party) increase market share, which seem to decrease it?

## Estimating Unit/Marginal Costs
- These firms pick their prices strategically, and price typically exceeds the true cost
- We want to understand how much the firms are making on each unit, in excess of the cost of creating it
- This is where economics begins: Do our markets maximize welfare? How much money are firms making by pricing strategically above cost? Are consumers exploited or not?
- The Amazon data are nice for data science purposes, but the same kind of analysis applies to education or epinephrine

## Inferring Cost
- How do we tie models to domain knowledge?
- Firm $j$ in period $t$ is maximizing its profits:
$$
\pi_{jt} = \underbrace{s_{jt}(p_{jt})}_{\text{Quantity Sold}} \quad \underbrace{(p_{jt}-c_{jt})}_{\text{Profit per Unit}}
$$
- That means that, at a maximum, it must be true that
$$
\pi_{jt}'(p_{jt}^*) = \underbrace{s_{jt}'(p_{jt}^*)(p_{jt}^*-c_{jt})}_{\text{Lost sales on the margin}} + \underbrace{s_{jt}(p_{jt}^*)}_{\text{Additional profits on remaining units sold}} = 0
$$
- If we re-arrange to solve for $c_{jt}$, we get
$$
\hat{c}_{jt} = p_{jt} + \dfrac{s_{jt}(p_{jt})}{s_{jt}'(p_{jt})}
$$
- This inverts demand and price to tell us about costs
- We can estimate the firm's unobserved, true cost of its product from demand behavior and prices
- Notice that price always exceeds marginal cost, here: Brands always price in excess of true cost

## Slope of Demand
- To keep things humane, I've pre-computed the derivative of the demand curve, like this:
$$
s_{jt}'(p_{jt}) = \frac{s_{jt}(p_{jt}+1)-s_{jt}(p_{jt}-1)}{2}
$$
- You can compute analytical derivatives using multinomial logistic regression, and GLM models in general provide us with dozens of ways to model and analyze how consumers respond to products

## 3. Market Fundamentals

Load `demand_4.parquet`. I've already computed $s_{jt}'(p_{jt})$ as `dprime`

- Compute the **unit cost** $\hat{c}_{jt}$ for each Brand in each period
- Compute the **mark-up** in each period, $markup_{jt} = p_{jt} - \hat{c}_{jt}$
- Compute the brand's **average profit** in each period, $ s_{jt} \times markup_{jt}$
- Make a kernel density plot of unit costs, mark-ups, and profits for each brand-year
- Which brands are most profitable? Which have the highest unit costs?
- Make scatter plots of price and unit cost, and average review and unit cost. Do you notice any patterns?

## 4. Welfare
The DOJ and FTC use these kinds of models for anti-trust investigations. Online commerce is the target of recent FTC regulatory actions, in particular towards Amazon. 

1. Do you think this market is relatively competitive?
2. Are the mark-ups sizeable? Are they worth pursuing regulatory or industrial policy over, or even an anti-trust lawsuit?
3. The bigger picture question is: "What is the market definition, and what products should be included?" The broader the market defintion, the more finely demand and sales are split, and the less power every firm appears to have (e.g. baseball; professional sports; entertainment). If you had to defend the brands we analyzed, what argument would you make?